# convolute solar spectrum and gas cross sections

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import prepare_sol_xsecs as pr
#import reload from ?
from importlib import reload


In [2]:
datapath= 'G:/Meine Ablage/LuftBlick/data/HR_database/'
projectpath='G:/Meine Ablage/LuftBlick/science-projects/ML-cal-retrieval/'


## solar spectrum

In [3]:
reload(pr)
wl_start = 300
wl_end = 900#500
F0_df = pr.get_all_F0(wl_start, wl_end)

In [4]:
dwl = 0.1
convoluted_df = pr.convolute_and_interpolate_spectrum(F0_df, wl_start, wl_end, dwl, 0.5, 'gaussian')

In [7]:
# save convoluted spectrum as pickle in projectpath
convoluted_df.to_pickle(projectpath+'/data/constants/solar_conv05.pkl')


In [9]:
# Plotting to verify
ax = plt.figure(figsize=(10, 6)).add_axes([0,0, 0.8, 0.8])
ax.plot(F0_df['wl'], F0_df['data'], '-',lw=.2,c='.5')
ax.plot(convoluted_df['wl'], convoluted_df['data'],'.-',c=(.1,.2,.4))
ax.set_xlim(320,920)

## gas cross sections / ODs

In [10]:
# table from manual (table20.txt, changed in newer version)
gas_table=pd.read_csv(projectpath+'data/'+'table20.txt',names=['gas','name','amount','d','Teff'],sep=' ',skiprows=1)
gas_table


In [27]:
gases_UVVIS = [
    'BrO',
    'HCHO',
    'HONO',
    'I2',
    'NO2',
    'O3',
    'SO2']

# 'O2',???

In [28]:
gases_UVVIS

In [29]:
reload(pr)
wl_start = 300
wl_end = 900

ods_df = pr.get_all_ods(gas_table, gases_UVVIS, wl_start, wl_end, 0)
ods_df.head()

In [34]:
reload(pr)
dwl = 0.1
ods_conv_df = pr.OD_convolution_and_interpolation(ods_df, wl_start, wl_end, dwl, slit_width_nm=0.5, slit_function='gaussian')
ods_conv_df.head()

In [35]:
# Plotting to verify
gas = 'NO2'
ax = plt.figure(figsize=(10, 6)).add_axes([0,0, 0.8, 0.8])
ax.plot(ods_df['wl'], ods_df[gas], '-',lw=.2,c='.5')
ax.plot(ods_conv_df['wl'], ods_conv_df[gas],'.-',c=(.1,.2,.4))

ax.plot(ods_conv_df['wl'], ods_conv_df['O3'],'.-',c=(.4,.2,.2))

ax.set_xlim(320,500)
ax.set_ylim(0,0.1)

In [36]:
# save convoluted spectrum as pickle in projectpath
ods_conv_df.to_pickle(projectpath+'/data/constants/gas_ods_conv05.pkl')

In [5]:
ods_conv_df = pd.read_pickle('../data/constants/gas_ods_conv05.pkl') 
ods_conv_df.head()

,wl,BrO,HCHO,HONO,I2,NO2,O2,O3,SO2
0,300.0,0.000049,0.000039,0.000009,0.000137,0.000841,0.000841,1.398054,0.006880
1,300.1,0.000065,0.000052,0.000012,0.000187,0.001143,0.001143,1.896130,0.008883
2,300.2,0.000076,0.000063,0.000016,0.000225,0.001384,0.001384,2.288004,0.010053
3,300.3,0.000081,0.000071,0.000018,0.000250,0.001538,0.001538,2.529327,0.010249
4,300.4,0.000082,0.000082,0.000020,0.000261,0.001619,0.001619,2.638861,0.009698


## SWIR

In [35]:
reload(pr)
wl_start = 1500
wl_end = 1750

F0_df = pr.get_all_F0(wl_start, wl_end)
#F0_df.head()
dwl = 0.1
convoluted_df = pr.convolute_and_interpolate_spectrum(F0_df, wl_start, wl_end, dwl, 0.6, 'gaussian')

In [39]:
# Plotting to verify
ax = plt.figure(figsize=(10, 6)).add_axes([0,0, 1, 0.2])
ax.plot(F0_df['wl'], F0_df['data'], '-',lw=.2,c='.5')
ax.plot(convoluted_df['wl'], convoluted_df['data'],'.-',c=(.1,.2,.4),ms=2,lw=.5)
#ax.set_xlim(320,420)

In [4]:
gases_IR = [
    'CO2',
    'CH4',
    'H2O']

In [12]:
reload(pr)
wl_start = 1500
wl_end = 1750
dwl = 0.1

ods_df = pr.get_all_ods(gas_table, gases_IR, wl_start, wl_end, 0)
ods_conv_df = pr.OD_convolution_and_interpolation(ods_df, wl_start, wl_end, dwl, slit_width_nm=0.6, slit_function='gaussian')
ods_conv_df.head()

In [13]:
dwl=.5
ods_conv_df = pr.OD_convolution_and_interpolation(ods_df, wl_start, wl_end, dwl, slit_width_nm=2, slit_function='gaussian')


In [14]:
ods_df.head()

In [30]:
sf = 20 # scale factor
ods_df_scaled = ods_df.copy()
ods_df_scaled.loc[:, ods_df_scaled.columns != 'wl'] *= sf
ods_conv_df_scaled1 = pr.OD_convolution_and_interpolation(ods_df_scaled, wl_start, wl_end, dwl, slit_width_nm=2, slit_function='gaussian')

# now scaled convoluted spectrum
ods_conv_df_scaled2 = ods_conv_df.copy()
ods_conv_df_scaled2.loc[:, ods_conv_df_scaled2.columns != 'wl'] *= sf


In [37]:
gas = 'CH4'
gas = 'CO2'

ax = plt.figure(figsize=(10, 6)).add_axes([0,0, 0.8, 0.5])
#ax.plot(ods_conv_df['wl'], ods_conv_df['CO2'], '-',lw=2,c='.5')
ax.plot(ods_conv_df_scaled1['wl'], ods_conv_df_scaled1[gas],lw=1,c=(.1,.2,.4))
ax.plot(ods_conv_df_scaled2['wl'], ods_conv_df_scaled2[gas],'--',c=(.1,.4,.8))

ax.set_xlim(1500,1750)
#ax.set_ylim(0,0.3)

In [38]:
ax = plt.figure(figsize=(10, 6)).add_axes([0,0, 0.8, 0.5])
ax.plot(ods_conv_df_scaled1['wl'], ods_conv_df_scaled1[gas]/ods_conv_df_scaled2[gas],lw=1,c=(.1,.2,.4))
ax.set_xlim(1500,1750)
ax.set_ylim(1-1e-2,1+1e-2)

In [59]:
ax = plt.figure(figsize=(10, 6)).add_axes([0,0, 0.8, 0.8])

# loop over gases
# each gas with different color
cs = plt.cm.tab10(np.linspace(0,2,10))#len(gases_IR))
for k, gas in enumerate(gases_IR):
    ax.plot(ods_conv_df['wl'], ods_conv_df[gas],'.-',c=cs[k],label=gas)

    ax.plot(ods_df['wl'], ods_df[gas], '-',lw=.1,c=cs[k])


ax.set_xlim(1500,1750)
ax.set_ylim(0,0.3)

In [1]:
ods_df.head()


In [ ]:
dwl=.5
ods_conv_df = pr.OD_convolution_and_interpolation(ods_df, wl_start, wl_end, dwl, slit_width_nm=2, slit_function='gaussian')

In [30]:
# Plotting to verify
gas = 'CO2'
ax = plt.figure(figsize=(10, 6)).add_axes([0,0, 0.8, 0.8])

#ax.plot(ods_df['wl'], ods_df[gas], '-',lw=.2,c='.5')
ax.plot(ods_conv_df['wl'], ods_conv_df[gas],'.-',c=(.1,.2,.4))

#ax.plot(ods_df['wl'], ods_df['CH4'], '-',lw=.2,c='.5')
ax.plot(ods_conv_df['wl'], ods_conv_df['CH4'],'.-',c=(.4,.2,.2),ms=2,lw=1)

ax.set_xlim(1600,1700)
ax.set_ylim(0,0.3)

# fancy plotting

In [78]:
import matplotlib as mpl
from matplotlib.colors import ListedColormap
import seaborn as sns

def bg_dark(facolor):
    gacolor='.5'
    #facolor='.2'
    #facolor=(.2,.2,.4)

    mpl.rcParams['figure.facecolor']= facolor
    mpl.rcParams['axes.facecolor'] = facolor
    mpl.rcParams['axes.linewidth'] = .5
    mpl.rcParams['axes.edgecolor'] = gacolor
    mpl.rcParams['axes.axisbelow'] = True
    mpl.rcParams['axes.labelcolor'] = gacolor


    mpl.rcParams['xtick.direction']='out'
    mpl.rcParams['ytick.direction']='out'

    mpl.rcParams['xtick.major.size']=0
    mpl.rcParams['ytick.major.size']=0

    mpl.rcParams['xtick.color']=gacolor
    mpl.rcParams['ytick.color']=gacolor

    mpl.rcParams['grid.color'] =gacolor
    mpl.rcParams['grid.linewidth'] = .1#.3
    mpl.rcParams['grid.linestyle'] = '-'

In [79]:
cmap=ListedColormap(["#DBFFFF","#B3F9FE","#92F1DF","#89E6B6","#99D786","#B4C353","#CDAC2B","#E0913D","#EB736A","#ED5096"])
#cmap.set_bad((.22,.22,.27))
sns.palplot([cmap(k) for k in range(10)])

In [95]:
dwl=.05
ods_conv2_df = pr.OD_convolution_and_interpolation(ods_df, wl_start, wl_end, dwl, slit_width_nm=.1, slit_function='gaussian')

In [98]:
bgcolor=(.14,.145,.2) #final s1 colors
bg_dark(bgcolor)

fig = plt.figure(figsize=(12, 6))
w=0.3
dw = 0.02
# loop over gases
# each gas with different color
cs = plt.cm.tab10(np.linspace(0,2,10))#len(gases_IR))
cs = (cmap(3),cmap(5),cmap(8))
for k, gas in enumerate(gases_IR):

    ax = fig.add_axes([k*(w+dw),0, w, 0.5])
    c = cs[k]
    c= cmap(6)
    ax.plot(ods_conv_df['wl'], ods_conv_df[gas],'.-',c=c,lw=1,ms=0)
    #ax.plot(ods_df['wl'], ods_df[gas], '-',c=c,lw=.05)
    ax.plot(ods_conv2_df['wl'], ods_conv2_df[gas], '-',c=c,lw=.05)

    for gas2 in gases_IR:
        if gas2 != gas:
            ax.plot(ods_conv_df['wl'], ods_conv_df[gas2],'.-',c='.7',lw=1,ms=0)

    if k>0:
        ax.set_yticklabels([])

    ax.set_xticks(np.arange(1500,1750,20))
    ax.set_xlim(1500,1750)
    ax.set_ylim(0,0.3)
    ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=5)
    ax.annotate(gas, xy=(0, 1.02), xycoords="axes fraction", ha="left", va="bottom", fontsize=12, color='.5')

In [112]:
bgcolor=(.14,.145,.2) #final s1 colors
bg_dark(bgcolor)

ax = plt.figure(figsize=(12, 6)).add_axes([0,0, 0.8, 0.7])

# loop over gases
# each gas with different color
cs = plt.cm.tab10(np.linspace(0,2,10))#len(gases_IR))
cs = (cmap(3),cmap(6),cmap(8))
for k, gas in enumerate(gases_IR):

    c = cs[k]
    #c= cmap(6)

    #ax.plot(ods_df['wl'], ods_df[gas], '-',c=c,lw=.05)
    ax.plot(ods_conv2_df['wl'], ods_conv2_df[gas], '-',c=c,lw=.05)

    ax.plot(ods_conv_df['wl'], ods_conv_df[gas],'.-',c=c,lw=2,ms=0)
    
    ax.set_xticks(np.arange(1500,1750,20))
    ax.set_xlim(1500,1750)
    ax.set_ylim(0,0.3)
    ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=9)
    ax.annotate(gas, xy=(k*.1, 1.02), xycoords="axes fraction", ha="left", va="bottom", fontsize=12, color=c)